In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Loading Data

Before any analysis can begin, data must be loaded into Python. pandas supports a wide range of file formats and data sources out of the box.

In [ ]:
import pandas as pd

## CSV Files

CSV (Comma-Separated Values) is one of the most common data exchange formats. Use `pd.read_csv()` to load CSV files.

```{tip}
In many European countries, CSV files use `;` as separator instead of `,`. Always check and set the `sep` parameter accordingly.
```

In [ ]:
# Load a CSV file with semicolon separator
df = pd.read_csv(
    "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatTempHum.csv",
    sep=";"
)
df.head()

### Useful `read_csv` parameters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `sep` | Column separator | `sep=";"` |
| `parse_dates` | Columns to parse as datetime | `parse_dates=["time"]` |
| `index_col` | Column to use as row index | `index_col="time"` |
| `usecols` | Only load specific columns | `usecols=["time", "FlatA_Temp"]` |
| `nrows` | Only read first N rows | `nrows=100` |
| `encoding` | File encoding | `encoding="utf-8"` |

In [ ]:
# Load with datetime parsing and index
df_parsed = pd.read_csv(
    "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatTempHum.csv",
    sep=";",
    parse_dates=["time"],
    index_col="time"
)
df_parsed.head()

In [ ]:
df_parsed.dtypes

## Loading from URLs

`pd.read_csv()` works directly with URLs — no need to download the file first.

In [ ]:
url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatElectricity.csv"

df_elec = pd.read_csv(url, sep=";")
df_elec.head()

## Excel Files

Use `pd.read_excel()` to load Excel files (`.xlsx`, `.xls`).

:::{admonition} openpyxl required
:class: note dropdown
Reading Excel files requires the `openpyxl` package. Install it with:

```bash
pip install openpyxl
```
:::

In [ ]:
# Example: reading an Excel file (not executed here)
# df_excel = pd.read_excel("path/to/file.xlsx", sheet_name="Sheet1")
# df_excel.head()

### Useful `read_excel` parameters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `sheet_name` | Name or index of sheet to read | `sheet_name="Data"` |
| `header` | Row number for column headers | `header=0` |
| `skiprows` | Rows to skip at the start | `skiprows=2` |
| `usecols` | Columns to read | `usecols="A:D"` |

## JSON Files

JSON (JavaScript Object Notation) is commonly used for data exchange, especially with web APIs and IoT devices. Many building monitoring systems export sensor data as JSON.

A JSON file typically contains a list of records (like rows) or a nested structure.

### Example: sensor readings as JSON

Imagine a building management system exports sensor readings in this format:

In [ ]:
import json
from io import StringIO

# Simulated JSON export from a building management system
json_string = """[
    {"timestamp": "2024-01-15 08:00", "sensor": "Office_101", "temperature": 21.3, "co2": 420},
    {"timestamp": "2024-01-15 09:00", "sensor": "Office_101", "temperature": 22.1, "co2": 580},
    {"timestamp": "2024-01-15 10:00", "sensor": "Office_101", "temperature": 22.8, "co2": 720},
    {"timestamp": "2024-01-15 11:00", "sensor": "Office_101", "temperature": 23.1, "co2": 850}
]"""

# Load JSON string into a DataFrame
df_json = pd.read_json(StringIO(json_string))
df_json

:::{admonition} What happens here?
:class: tip dropdown

1. The JSON string contains a **list of objects** — each object is one measurement with timestamp, sensor ID, temperature, and CO2 level
2. `pd.read_json()` automatically converts each object into a **row** and each key into a **column**
3. `StringIO()` wraps the string so pandas reads it like a file

To load a JSON **file** from disk instead: `df = pd.read_json("sensor_data.json")`
:::

### Loading JSON from a web API

Many data providers offer REST APIs that return JSON. Use the  library to fetch the data:

In [ ]:
# Example: fetching sensor data from an API
# import requests
#
# response = requests.get("https://api.building-system.com/sensors/Office_101/readings")
# data = response.json()  # returns a Python list/dict
# df_api = pd.DataFrame(data)
# df_api.head()

## Saving Data

After processing, data can be saved back to various formats.

In [ ]:
# Save to CSV
# df.to_csv("output.csv", sep=";", index=False)

# Save to Excel
# df.to_excel("output.xlsx", index=False)

```{tip}
Use `index=False` when saving to prevent pandas from writing the row index as an extra column.
```

## Using pyedautils for Data I/O

I/O stands for **Input/Output** — reading data into Python (input) and writing data to files (output).

The `pyedautils` package provides `save_data()` and `load_data()` functions that simplify saving and loading DataFrames with preserved data types.

:::{admonition} ModuleNotFoundError?
:class: warning dropdown
If you get a `ModuleNotFoundError: No module named 'pyedautils'`, install it first:

```bash
pip install pyedautils
```

Then restart the kernel and run the cell again.
:::

In [ ]:
from pyedautils.data_io import save_data, load_data

# Create a sample DataFrame
df_sample = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01", periods=5, freq="h"),
    "temperature": [21.3, 21.5, 22.0, 22.4, 22.1],
    "humidity": [45.0, 44.8, 43.5, 42.0, 43.2]
})

# Save the DataFrame
save_data(df_sample, "my_sensor_data.pkl")
print("Data saved.")

:::{admonition} save_data() and load_data() — full reference
:class: tip dropdown

**`save_data(data, file_path, index=False)`** saves a DataFrame to a file. The format is determined by the file extension:

| Extension | Format | Best for |
|---|---|---|
| `.csv` | CSV (text) | Data exchange with Excel or other tools |
| `.pkl` | Pickle (binary) | Fast intermediate storage, preserves all data types |
| `.pklz` | Compressed Pickle | Same as pickle but smaller file size |
| `.json` | JSON (text) | Data exchange with web APIs or JavaScript |

The `index` parameter controls whether the DataFrame index is written to the file (default: `False`).

**`load_data(file_path)`** loads data back from any of the above formats. The format is automatically detected from the file extension. For CSV files, the separator is auto-detected.

**Examples:**

```python
# Save as CSV (readable in Excel)
save_data(df, "output/measurements.csv")

# Save as pickle (fast, preserves dtypes)
save_data(df, "output/measurements.pkl")

# Save as compressed pickle (smaller file)
save_data(df, "output/measurements.pklz")

# Save with index column
save_data(df, "output/measurements.csv", index=True)

# Load any format back
df = load_data("output/measurements.pkl")
```

**Why pickle instead of CSV?**
- **Data types preserved** — datetime, float, int, categorical all stay intact. CSV converts everything to text
- **Faster** — binary format is much quicker to read/write, especially for large datasets
- **Exact reproduction** — the loaded DataFrame is identical to the saved one

Use CSV for sharing data with others, pickle for your own intermediate storage.
:::

In [ ]:
# Load it back
df_loaded = load_data("my_sensor_data.pkl")
df_loaded

In [ ]:
# Verify: dtypes are preserved (timestamp is still datetime, not a string)
df_loaded.dtypes